# FreshNet training (Colab or local)

This notebook trains `DualPathFreshNet` on a dataset **you provide**. It does not ship weights, accuracy numbers, or a verified 70,000-image count.

Rules:
- Outputs are visual freshness estimates, not food-safety results.
- Do not invent a Medium class for a two-class dataset.
- Do not paste fabricated metrics into this notebook.

## 1. Runtime

In Colab: **Runtime → Change runtime type → GPU** if you have a GPU quota. CPU also works, only slower.

Install PyTorch using the command from https://pytorch.org/get-started/locally/ if the next cell's default CPU wheels are wrong for your accelerator.

In [ ]:
import sys
print("Python", sys.version)

# Default CPU install for a clean Colab/local start. Replace with the official
# CUDA command from pytorch.org when you are on an NVIDIA GPU runtime.
%pip install -q torch torchvision pillow

## 2. Attach the dataset

Upload or mount an `ImageFolder` tree:

```text
data/raw/your-dataset/Fresh/...
data/raw/your-dataset/Medium/...
data/raw/your-dataset/Spoiled/...
```

Include `Medium` only if those labels exist in the source dataset. Then set `DATA_DIR` and `NUM_CLASSES` to match the folders you actually have.

In [ ]:
from pathlib import Path

DATA_DIR = Path("data/raw/your-dataset")
NUM_CLASSES = 3  # must equal the number of class folders

class_dirs = sorted(p for p in DATA_DIR.iterdir() if p.is_dir()) if DATA_DIR.is_dir() else []
print("Class folders:", [p.name for p in class_dirs])
if not class_dirs:
    print("Dataset not found. Training cells below will fail until DATA_DIR exists.")

## 3. Train with the repo CLI

If this notebook is running inside the FreshNet repo, call `ml/train.py`. In Colab, upload `ml/dual_path.py` and `ml/train.py` next to the notebook or clone your private copy of the repo first.

In [ ]:
import subprocess
from pathlib import Path

train_script = Path("ml/train.py")
if not train_script.exists():
    raise FileNotFoundError("ml/train.py is missing. Open this notebook from the FreshNet repo or upload ml/.")

completed = subprocess.run(
    [
        "python",
        str(train_script),
        "--data-dir",
        str(DATA_DIR),
        "--num-classes",
        str(NUM_CLASSES),
        "--epochs",
        "5",
    ],
    check=False,
)
print("exit code", completed.returncode)

If training succeeds, download `artifacts/checkpoints/freshnet.pt` into your local `artifacts/checkpoints/` directory. The website will keep showing **Model not trained** until that file is present and a later stage loads it.